# How the dashboard is put together

The companion document to `app.py`. It records the reasoning, in the order the decisions were made: what kind of market gold is, therefore which questions matter, therefore which sources, therefore which visual grammar, therefore this layout. Every code cell runs offline against the sample data (or regenerates it), so each claim about a derived series can be checked here.

## 1. The market structure that dictates the design

Above-ground gold stock is ~216,000 t. Annual mine output is ~3,600 t — 1.6% of stock, price-inelastic, sold at market regardless of level. Gold is therefore a stock equilibrium: the entire existing hoard is potential supply, and the price is whatever makes current holders content to keep holding. Production and consumption tell you almost nothing; the informative questions are **who holds the stock, on what motive, and in which direction stock is transferring between holder classes**.

Since 2022 the relevant holder classes are two: Western financial holders, for whom gold is a zero-coupon perpetual real asset priced off the real yield; and official-sector holders swapping freezable claims on foreign sovereigns for an asset with no issuer. The largest of the second class conceals its purchases. The dashboard instruments one term of that structure per panel.

## 2. Question list → panels

The specification is the question list; a panel with no question was cut.

| # | Question (answerable in <5 s) | Panel | Tab |
|---|---|---|---|
| 1 | Has the price detached from real yields, and by how much? | P1 price vs yield-implied level | Declared |
| 2 | Is the yield-sensitive holder buying this move or selling into it? | P2 GLD+IAU tonnes | Declared |
| 3 | Which central banks report accumulating, and how fast? | P3 trailing-12-month change | Declared |
| 4 | How much official buying goes unreported each quarter? | P4 IMF-reported vs WGC estimate | Inferred |
| 5 | Where are the bars going; is the float replenished? | P5 Swiss exports + mine reference lines | Inferred |
| 6 | What changed since I last looked? | P6 KPI strip | both |

The tab split is the thesis: **Declared** is what the system reports about itself; **Inferred** is what physical flows reveal that reporting omits. Every Inferred chart reconciles against something Declared.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# -- design tokens (mirrored in design.py) ------------------------------------
PAPER, INK, FAINT, GHOST, GOLD = "#fbfbf8", "#1c1c18", "#8c8c84", "#d9d9d2", "#a3780a"
HUES = {"China":"#b3392e","China + HK":"#b3392e","Russia":"#2e5fa3","Iran":"#2e7d5b",
        "India":"#c98a1b","Turkey":"#8e4a8b","UK + US":"#5b6b7a","West":"#5b6b7a",
        "UAE + Gulf":"#b9b9b0","other":"#d9d9d2"}
REST = "#b9b9b0"
hue = lambda e: HUES.get(e, REST)
EVENTS = [("2022-02-28","freeze"),("2022-06-26","G7 ban"),("2025-06-13","Iran war"),
          ("2025-11-01","CBR sells"),("2026-05-15","escalation")]
plt.rcParams.update({"figure.facecolor":PAPER,"axes.facecolor":PAPER,
    "axes.edgecolor":GHOST,"axes.grid":False,"font.size":9,
    "axes.spines.top":False,"axes.spines.right":False})

# -- load sample data, or regenerate it so this notebook is self-contained ----
DATA = Path("data")
if not (DATA/"price.csv").exists():
    import subprocess, sys
    subprocess.run([sys.executable, "ingest.py", "--synthetic"], check=True)
price    = pd.read_csv(DATA/"price.csv", parse_dates=["date"])
etf      = pd.read_csv(DATA/"etf.csv", parse_dates=["date"])
declared = pd.read_csv(DATA/"declared.csv", parse_dates=["date"])
gap      = pd.read_csv(DATA/"gap.csv", parse_dates=["quarter_end"])
swiss    = pd.read_csv(DATA/"swiss.csv", parse_dates=["date"])
print("loaded:", {p.stem: len(pd.read_csv(p)) for p in sorted(DATA.glob("*.csv"))})

## 3. Why these sources

Each choice is source-over-alternatives, decided by four constraints: licence, cadence, lag, and whether it is primary.

- **P1 price — COMEX front-month futures (GC=F), not the LBMA benchmark.** The London price sits behind an ICE Benchmark Administration licence (historical data withdrawn from public redistribution in March 2025). Futures are free, daily, continuous — and honest labelling ("futures") costs nothing analytically at monthly resolution.
- **P1 opportunity cost — FRED DFII10.** The canonical 10-year real yield, keyless CSV endpoint. The panel's content is the residual after regressing price on it over pre-2022 data.
- **P2 — GLD + IAU issuer bar lists, not WGC ETF data.** Issuer files are primary, daily and free; the WGC series is monthly behind a login. Two funds carry most Western tonnage, and a biased-but-daily proxy beats a complete-but-monthly one for a monitoring panel.
- **P3 — IMF International Financial Statistics.** The only comprehensive cross-country official series. Voluntary reporting and the 1–3 month lag are features here, since P4 measures exactly that gap.
- **P4 — WGC Gold Demand Trends estimate vs the IMF aggregate.** The only public series incorporating estimated unreported buying. Four numbers a year: hand-entered, deliberately — automating a login scrape for four observations is negative-value engineering.
- **P5 — Swiss-Impex, tariff heading 7108.** One free monthly pull with a ~3-week lag captures the London→Ticino→Asia conversion circuit, because Switzerland is the form-conversion hub between the 400 oz wholesale market and Asian kilobar demand (roughly a third to a half of refined supply transits it). UN Comtrade covers more pairs at months more lag; it was cut with the rest of the long tail.
- **Mine reference lines — USGS Mineral Commodity Summaries.** Two constants a year, hand-entered. Mine supply is slow, price-taking flow with no information about the marginal holder, so it appears as context lines on P5, never as a panel.

In [ ]:
constraints = pd.DataFrame([
 ("P1","COMEX GC=F + FRED DFII10","daily","1 day","free, keyless","futures, not spot"),
 ("P2","GLD + IAU bar lists","daily","1 day","free","2 funds ≈ most, not all, Western tonnage"),
 ("P3","IMF IFS monetary gold","monthly","1-3 mo","free API","voluntary reporting"),
 ("P4","WGC GDT vs IMF aggregate","quarterly","~1 mo","free login","estimate, hand-entered"),
 ("P5","Swiss-Impex HS 7108","monthly","~3 wk","free CSV","last transit country, not origin"),
 ("ref","USGS mine supply","annual","months","free","smeared to monthly rate"),
], columns=["panel","source","cadence","lag","access","known bias"])
constraints

## 4. Derived series

The raw series are inputs; these four are the dashboard's content. Each inherits the slowest input's cadence and lag.

1. **Sanctions premium** (P1 wedge) = price − (a + b·real yield), with (a, b) fit on pre-2022 data only. The wedge is the footprint of the yield-indifferent buyer.
2. **Unreported gap** (P4 shading) = WGC estimate − IMF-reported, per quarter.
3. **Eastward share** (P6 tile) = Swiss exports to China+HK, India, Turkey ÷ total Swiss exports.
4. **Replenishment context** (P5 toggle) = two reference rates drawn against the stacked total: all mine supply (~300 t/mo) and the ~200 t/mo of it that enters the Western circuit (China, Russia, Uzbekistan and Kazakhstan production never does). Swiss exports are one hub's view of the drain, so the comparison is directional context rather than a closed balance; a shrinking free float raises the convexity of price to any Western demand shock — the melt-up mechanism.

In [ ]:
pre = price[price.date < "2022-01-01"]
b, a = np.polyfit(pre.real_yield_pct, pre.price_usd_oz, 1)
premium = price.price_usd_oz - (a + b*price.real_yield_pct)
gap["unreported_t"] = gap.wgc_estimate_t - gap.imf_reported_t
east = (swiss[swiss.destination.isin(["China + HK","India","Turkey"])]
        .groupby("date").tonnes.sum())
print(f"pre-22 fit: price ≈ {a:,.0f} {b:+,.0f}·y   |  latest premium: {premium.iloc[-1]:,.0f} $/oz")
print(f"latest quarter unreported: {gap.unreported_t.iloc[-1]:+.0f} t "
      f"({gap.unreported_t.iloc[-1]/gap.wgc_estimate_t.iloc[-1]:.0%} of estimated official demand)")
print(f"eastward flow, latest month: {east.iloc[-1]:.0f} t "
      f"= {east.iloc[-1]/200:.0%} of Western-circuit replenishment, via this one hub alone")

## 5. The visual grammar

Three fixed mappings, enforced on every panel:

| mapping | rule | carries |
|---|---|---|
| entity → hue | one colour per country, all panels | identity across panels |
| epistemic status → line style | **solid = reported, dashed = estimated** | the thesis itself |
| data class → mark | lines = levels, bars = period changes, areas = composition | quantity semantics |

The second mapping is the design's centre: the Declared/Inferred split exists in the visual grammar, so P4's dashed estimate against solid bars reads without a legend. The demonstration below draws the same two series with and without the grammar.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 2.6), sharey=True)
for k, (dash, title) in enumerate([("-", "no grammar: two equal claims"),
                                   ("--", "grammar: a report and an estimate")]):
    ax[k].plot(gap.quarter_end, gap.imf_reported_t, color=INK, lw=1.6)
    ax[k].plot(gap.quarter_end, gap.wgc_estimate_t, color=INK, lw=1.4, ls=dash)
    if k == 1:
        ax[k].fill_between(gap.quarter_end, gap.imf_reported_t, gap.wgc_estimate_t,
                           color=GOLD, alpha=.16)
    ax[k].set_title(title, fontsize=9, loc="left")
fig.tight_layout(); plt.show()

## 6. Gestalt principles → concrete decisions

- **Common region.** One bordered container on the whole page: the KPI strip (`st.container(border=True)`). The border means "ambient state of the system"; nothing else earns one.
- **Proximity.** Panels are grouped by whitespace only. A box around every chart would spend the grouping signal on nothing.
- **Similarity.** The three mappings above. Colour repetition across panels is what lets the eye track China from P3 to P5 without reading a legend.
- **Continuity.** Every time panel shares one x-domain and one plot width, stacked in a single column; the event rules (freeze, G7 ban, Iran war, CBR sales, escalation) sit at identical x-positions, so the eye carries an event down the column. This is the argument for single-column over a grid.
- **Figure-ground.** Focal entities saturated, rest-of-world grey, gridlines barely visible. In the per-bank detail, the bank's own series is coloured and the others are absent rather than ghosted — on a phone, absence beats ghosting.
- **Prägnanz / hierarchy.** One dominant chart per tab at roughly double height (P1, P4), directly under the header. No panel carries more than three saturated series; the fifteen-bank comparison is a sorted bar plus small multiples, never spaghetti.
- **Closure.** KPI tiles carry deltas and (in the HTML mockup) axis-free sparklines; the eye completes the trend at that size.

Two demonstrations: continuity via aligned event rules, and figure-ground against the spaghetti alternative.

In [ ]:
ev = [pd.Timestamp(d) for d, _ in EVENTS]
fig, ax = plt.subplots(2, 1, figsize=(8, 3.4), sharex=True)
ax[0].plot(price.date, price.price_usd_oz, color=GOLD, lw=1.8)
ax[1].stackplot(swiss.date.unique(),
    *[swiss[swiss.destination == d].tonnes.values for d in
      ["other","UAE + Gulf","UK + US","Turkey","India","China + HK"]],
    colors=[hue(d) for d in ["other","UAE + Gulf","UK + US","Turkey","India","China + HK"]],
    alpha=.8)
for a in ax:
    for x in ev: a.axvline(x, color=GHOST, lw=1, zorder=0)
ax[0].set_title("continuity: the same event rules on every panel", fontsize=9, loc="left")
fig.tight_layout(); plt.show()

In [ ]:
t12 = (declared[declared.date > declared.date.max() - pd.DateOffset(months=12)]
       .groupby("country").net_purchases_t.sum().sort_values())
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for c in t12.index:                                   # spaghetti: rejected
    d = declared[declared.country == c]
    ax[0].plot(d.date, d.net_purchases_t.cumsum(), lw=1)
ax[0].set_title("rejected: 15 series, one axes", fontsize=9, loc="left")
ax[1].barh(t12.index, t12.values, color=[hue(c) for c in t12.index])
ax[1].set_title("kept: sorted change, entities in their hues", fontsize=9, loc="left")
fig.tight_layout(); plt.show()

## 7. Form factor and layout

The first layout decision was the device, and it was made before any panel: this is a phone-first monitor read in the gaps of a day, desktop-tolerant rather than desktop-optimised. Consequences, in order:

1. **Single column.** Continuity (shared x, aligned rules) survives stacking; a grid does not. Streamlit's `layout="centered"` gives the column; `st.columns` is used only inside the KPI strip, where wrapping to 3×2 is acceptable.
2. **Stacking order = narrative order.** Declared reads price → investors → official sector; Inferred reads gap → flow. The order on a phone is the argument's order.
3. **Tabs, not multipage.** `st.tabs` keeps shared filters and one load; Streamlit's sidebar (the multipage navigation) collapses behind a hamburger on mobile and would hide the page split entirely.
4. **Glanceability budget (Few).** Six KPI numbers with as-of dates; one dominant chart per tab; captions instead of axis clutter.
5. **Interaction budget.** One toggle (mine replenishment, default off — explanatory depth, never ambient state) and one expander (per-bank detail). Hover is treated as unavailable.

What was cut, for manageability: Shanghai premium, UK HMRC proxy, CFTC positioning (the three stretch items, in value order), and the whole long tail — Hong Kong and Chinese customs, UN Comtrade, MOEX, Tehran prices, COMEX/LBMA inventories, lease rates. Five ingesters is a weekend; twenty-five is a maintenance job.

## 8. Build order and acceptance tests

Order actually followed: P4 first (depends on nothing automated, forces the reconciliation logic), then P1, P2, P5, P3, KPI strip last (derived from the rest). Ingestion is separated from presentation — `ingest.py` writes CSV vintages, `app.py` only reads — because customs and IMF sources revise silently and the app must never block on a remote fetch.

Tests the dashboard must pass:
1. every panel answers its question without a legend lookup of more than six entities;
2. every panel shows its as-of date;
3. page load under three seconds on a phone;
4. every derived number in the KPI strip reproducible from this notebook against the stored vintage;
5. the question test per panel (read it, state the answer, check the source) and the squint test on P1 and P4.